# Kurzübersicht: Iterative Verhältniswahl

Dieses Notebook zeigt den vollständigen Ablauf einer iterativen Verhältniswahl an einem kleinen, handnachvollziehbaren Beispiel (5 Wahlkreise, 3 Parteien).

**Ablauf:**
1. Iteration 1 — keine Partei erreicht die Stimmenmehrheit
2. Iteration 2 — injizierte Wahrscheinlichkeiten, wieder kein Gewinner
3. Koalitionsbildung — zwei Parteien bilden eine Koalition und gewinnen
4. Sitzverteilung und Wahlkreiszuordnung

Für echte Wahldaten: [`bundestagswahl.ipynb`](bundestagswahl.ipynb)  
Für Details zu einzelnen Schritten: die jeweiligen Schritt-Notebooks ([Übersicht](../../docs/source/de/einfuehrung.md))

In [ ]:
import pandas as pd
from matplotlib import pyplot as plt
from IPython.display import display

from ipres import (
    Election, ElectionConfig, ConstituenciesConfig,
    ElectionEvaluator, VoteMatrixAnalyzer,
    SuperMajorityMargin, MarginUnit,
)

cc = ConstituenciesConfig.from_dataframe(pd.DataFrame({
    'constituency_name': ['WK1', 'WK2', 'WK3', 'WK4', 'WK5'],
    'constituency_size': [100_000] * 5,
}))

config = ElectionConfig(
    constituencies_config=cc,
    participating_parties=['A', 'B', 'C'],
    parliament_majority_margin=SuperMajorityMargin(5.0, MarginUnit.PERCENT),
    seed=42,
)

election = Election(electionConfig=config)

print(f"Parlamentssitze:  {config.parliamentarySeats}")
print(f"Regierungssitze:  {config.getParliamentMajoritySeats()}")
print(f"Ballotmehrheit:   {config.getBallotMajorityPercent():.1f} %  (mindestens nötig, um eine Iteration direkt zu gewinnen)")

---

## Iteration 1

Wir injizieren konkrete Stimmzahlen, damit die Ergebnisse handnachvollziehbar sind.

| Wahlkreis | A  | B  | C  | Σ   |
|-----------|----|----|-----|-----|
| WK1       | 48 | 32 | 20 | 100 |
| WK2       | 50 | 30 | 20 | 100 |
| WK3       | 46 | 34 | 20 | 100 |
| WK4       | 51 | 29 | 20 | 100 |
| WK5       | 49 | 31 | 20 | 100 |
| **Σ**     | **244** | **156** | **100** | **500** |
| **Anteil** | **48,8 %** | **31,2 %** | **20,0 %** | |

Keine Partei erreicht die Ballotmehrheit → kein Gewinner. C scheidet aus (A+B decken > 2/3 der Stimmen ab).

In [ ]:
votes = {
    'WK1': {'A': 48, 'B': 32, 'C': 20},
    'WK2': {'A': 50, 'B': 30, 'C': 20},
    'WK3': {'A': 46, 'B': 34, 'C': 20},
    'WK4': {'A': 51, 'B': 29, 'C': 20},
    'WK5': {'A': 49, 'B': 31, 'C': 20},
}

iteration1 = election.start(votes=votes)
display(iteration1.show_results_table(styler=True))
fig = iteration1.plot_vote_share_pie(title="Iteration 1: Stimmanteile")
display(fig)
plt.close(fig)

print(f"Gewinner? {election.hasWinner()}")

---

## Iteration 2 — Injizierte Wahrscheinlichkeiten

C scheidet aus. Wir geben an, wie C-Wähler voraussichtlich abstimmen werden.
Die injizierten Wahrscheinlichkeiten geben die Stimmanteile direkt vor und werden automatisch normiert.
Auch jetzt erreicht keine Partei die Ballotmehrheit.

In [ ]:
i2input = iteration1.getNextRoundInput()
print(f"Verbleibende Parteien: {list(i2input.contestants.keys())}")

# C-Wähler verteilen sich mehrheitlich zu A, Minderheit zu B
i2input.probabilities = {"A": 49, "B": 51}

iteration2 = election.runNextIteration(i2input)
display(iteration2.show_results_table(styler=True))
fig = iteration2.plot_vote_share_pie(title="Iteration 2: Stimmanteile")
display(fig)
plt.close(fig)

print(f"Gewinner? {election.hasWinner()}")

---

## Koalitionsbildung

Keine Partei erreicht die Mehrheit. A und B bilden eine Koalition.
Die Stimmen beider Parteien werden zusammengefasst — die Koalition übersteigt die Ballotmehrheit.

In [ ]:
iteration2.formCoalition("A+B", ["A", "B"])

display(iteration2.show_results_table(styler=True))
fig = iteration2.plot_vote_share_pie(title="Iteration 2 nach Koalitionsbildung")
display(fig)
plt.close(fig)

print(f"Gewinner? {election.hasWinner()}")
print(f"Wahlgewinner: {election.getWinner().name}")

---

## Sitzverteilung

Der `ElectionEvaluator` führt automatisch alle drei Auswertungsschritte durch:
Mandatszuteilung → Wahlkreisanzahlzuordnung → Wahlkreiszuordnung.

Details: [`wahlauswertung.ipynb`](wahlauswertung.ipynb)

In [ ]:
evaluator = ElectionEvaluator(seed=config.seed)
result = evaluator.evaluate(election)

display(result.get_seat_distribution_table())
fig = result.plot_seat_share_pie()
display(fig)
plt.close(fig)

---

## Abstimmungsmatrix und Wahlkreiswichtigkeit

Die relative Stimmmatrix zeigt den Stimmenanteil jeder Partei pro Wahlkreis (normiert).
Die Wichtigkeitsmatrix bewertet, für welche Partei ein Wahlkreis besonders bedeutsam ist — Grundlage der Wahlkreiszuordnung.

In [ ]:
vm_analyzer = VoteMatrixAnalyzer(election.getFirstIteration().vote_matrix.getVotes())
display(vm_analyzer.show_relative_vote_matrix(styler=False))
display(vm_analyzer.show_constituency_importance_matrix(styler=False, decimals=4))

---

## Wahlkreiszuordnung

Jeder Wahlkreis wird der Partei zugeordnet, für die er relativ am wichtigsten ist.

Details: [`wahlauswertung.ipynb`](wahlauswertung.ipynb)

In [ ]:
display(result.get_constituency_summary_table())
display(result.get_constituency_assignment_table(sort_by='constituency'))